# MERIT Hydro HAND (landslide input)

**Preferred batch path:**

```bash
python transformation/merit_hydro/extract_hand.py --site plymouth
```

Height Above Nearest Drainage (`hnd`) at 90 m → `layers.hand` under `landslide_hazard/sites/<city>/data/input/`.


Height Above Nearest Drainage for `LANDSLIDES_SITE` at 90 m. Low HAND amplifies drainage-convergence risk in the landslide hazard score.


In [ ]:
# Site configuration — transformation/landslide_hazard city configs + model defaults
import os
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
_LANDSLIDE_HAZARD = None
for _candidate in [_HERE, *_HERE.parents]:
    _probe = _candidate / "landslide_hazard" if _candidate.name != "landslide_hazard" else _candidate
    if (_probe / "site_config.py").is_file() and (_probe / "config" / "sites").is_dir():
        _LANDSLIDE_HAZARD = _probe
        break
if _LANDSLIDE_HAZARD is None:
    raise FileNotFoundError("Could not locate transformation/landslide_hazard from notebook cwd")

sys.path.insert(0, str(_LANDSLIDE_HAZARD))
from site_config import load_site_config

LANDSLIDE_HAZARD_ROOT = _LANDSLIDE_HAZARD

# Set the city here (edit this line). That value wins for interactive runs.
# Use None to fall back to env LANDSLIDES_SITE (default porto_alegre).
SITE_SLUG = "plymouth"  # or: "porto_alegre" | "edina" | "richfield" | "rochester" | "apple_valley" | None
if SITE_SLUG is None:
    SITE_SLUG = os.environ.get("LANDSLIDES_SITE", "porto_alegre")
SITE_CONFIG = load_site_config(SITE_SLUG, LANDSLIDE_HAZARD_ROOT)
SITE_ROOT = SITE_CONFIG["paths_abs"]["site_root"]
INPUT_DIR = SITE_CONFIG["paths_abs"]["data_input"]
INTERMEDIATE_DIR = SITE_CONFIG["paths_abs"]["data_intermediate"]
OUTPUT_DIR = SITE_CONFIG["paths_abs"]["data_output"]
OUT_ROOT = SITE_CONFIG["paths_abs"]["out"]
CACHE_DIR = SITE_CONFIG["paths_abs"]["cache"]
STYLES_DIR = SITE_CONFIG["paths_abs"]["styles"]
OUTPUT_PREFIX = SITE_CONFIG["output_prefix"]
SEASON = SITE_CONFIG["season"]
SEASON_LABEL = SITE_CONFIG["season_label"]
START_YEAR = int(SITE_CONFIG["start_year"])
END_YEAR = int(SITE_CONFIG["end_year"])
DW_YEAR = int(SITE_CONFIG.get("dw_year", 2023))
HAZARD_CFG = SITE_CONFIG["hazard"]
PUBLISH_CFG = SITE_CONFIG.get("publish", {})
BAIRRO_CFG = SITE_CONFIG.get("bairro", {})
MODEL_CONFIG_PATH = SITE_CONFIG["model_config_path"]
LAYER_FILES = SITE_CONFIG["layers"]
OUTPUT_FILES = SITE_CONFIG["outputs"]

for _p in (INPUT_DIR, INTERMEDIATE_DIR, OUTPUT_DIR, OUT_ROOT, CACHE_DIR, STYLES_DIR):
    Path(_p).mkdir(parents=True, exist_ok=True)

print(f"Landslide hazard site: {SITE_CONFIG['display_name']} ({SITE_SLUG})")
print(f"Config: {SITE_CONFIG['config_path']}")
print(f"Model defaults: {MODEL_CONFIG_PATH}")
print(f"Season: {SEASON_LABEL} {START_YEAR}-{END_YEAR}")
print(f"Inputs -> {INPUT_DIR}")


## 0. Earth Engine + ROI


In [ ]:
import ee
ee.Initialize(project='eecc-maureen')

# Site ROI: use the site polygon when available; fall back to bbox.
import json
import ee


def load_site_roi() -> ee.Geometry:
    boundary_path = SITE_CONFIG["boundary_path_abs"]
    if boundary_path.exists():
        data = json.loads(boundary_path.read_text())
        if data.get("type") == "FeatureCollection":
            features = [
                ee.Feature(ee.Geometry(feature["geometry"]), feature.get("properties", {}))
                for feature in data.get("features", [])
                if feature.get("geometry")
            ]
            if features:
                return ee.FeatureCollection(features).geometry()
        if data.get("type") == "Feature":
            return ee.Geometry(data["geometry"])
        if data.get("type") in {"Polygon", "MultiPolygon", "GeometryCollection"}:
            return ee.Geometry(data)
    return ee.Geometry.Rectangle(SITE_CONFIG["bbox"])


roi = load_site_roi()
print(f"ROI loaded for {SITE_CONFIG['display_name']} from {SITE_CONFIG['boundary_path_abs']}")


## 1. Export HAND


In [ ]:
# MERIT Hydro HAND → landslide site input/
from gee_local_export import export_image_to_input

merit = ee.Image("MERIT/Hydro/v1_0_1")
hnd = (
    merit.select("hnd")
    .clip(roi)
    .reproject(crs="EPSG:4326", scale=90)
    .toFloat()
)

export_image_to_input(
    hnd,
    filename=LAYER_FILES["hand"],
    region=roi,
    scale=90,
    input_dir=INPUT_DIR,
    crs="EPSG:4326",
    description=Path(LAYER_FILES["hand"]).stem,
    drive_folder="gee_exports",
)
print("Exports complete. Files land under:", INPUT_DIR)


## 2. Inspect


In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt

hand_tif = INPUT_DIR / LAYER_FILES['hand']
assert hand_tif.exists(), f'HAND not found: {hand_tif}'

with rasterio.open(hand_tif) as src:
    arr = src.read(1)
    nd = src.nodata
    valid = arr[np.isfinite(arr) & (arr != (nd if nd is not None else -9999))]
    print(f'HAND 90m: shape={src.shape} range=[{valid.min():.1f}, {valid.max():.1f}] m')

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(valid[valid < 100], bins=60, color='steelblue', edgecolor='none')
ax.axvline(50, color='orange', linestyle='--', label='50 m threshold')
ax.set_xlabel('HAND (m)')
ax.set_title(f"{SITE_CONFIG['display_name']} — HAND")
ax.legend()
plt.tight_layout(); plt.show()
